# Estudo global do Radar Financeiro

Este notebook consolida a leitura global do Radar em quatro estudos:

1. Participacao no Radar e motivos de nao participacao.
2. Resultado final, temas prioritarios e empates no topo.
3. Resultado financeiro do mes e conexao com temas prioritarios.
4. Perfil do cliente e conexao com temas prioritarios.

A primeira celula prepara a base analitica reutilizada pelos quatro estudos.

In [ ]:
%%spark

from pyspark.sql import functions as F
from pyspark.sql.window import Window

tabela_origem = globals().get("tabela_spark", "sbx_t2i2016.v1_ana_edu_fin_cli")

df = spark.table(tabela_origem).dropDuplicates(["CD_CLI"])

def texto_normalizado(coluna, padrao="A CLASSIFICAR"):
    return F.upper(F.trim(F.coalesce(F.col(coluna).cast("string"), F.lit(padrao))))

def pct_sobre(coluna_qt, denominador):
    if denominador and denominador > 0:
        return F.round(F.col(coluna_qt) / F.lit(denominador) * 100, 2)

    return F.lit(0.0)

def contar_clientes(df_in, agrupadores, denominador, nome_percentual, ordenar_por=None):
    resultado = (
        df_in
        .groupBy(*agrupadores)
        .agg(F.countDistinct("CD_CLI").alias("Clientes"))
        .withColumn(nome_percentual, pct_sobre("Clientes", denominador))
    )

    if ordenar_por:
        resultado = resultado.orderBy(*ordenar_por)

    return resultado

def mostrar(df_saida, n=200):
    df_saida.show(n, truncate=False)

temas_pontuacao = [
    ("Categorizacao de gastos", "NR_PONT_CATEG", 10),
    ("Gestao do Orcamento", "NR_PONT_ORC", 20),
    ("Consumo Planejado", "NR_PONT_CONS", 30),
    ("Formacao de Reserva", "NR_PONT_RES", 40),
    ("Uso Consciente do Credito", "NR_PONT_CRED", 50),
]

movimentacao_minima = (
    (F.coalesce(F.col("QT_TRANS_TOTAL"), F.lit(0)) > 0)
    & (F.coalesce(F.col("QT_TRANS_ENT"), F.lit(0)) > 0)
    & (F.coalesce(F.col("QT_TRANS_SAI"), F.lit(0)) > 0)
    & (F.coalesce(F.col("VL_TRANS_ENT"), F.lit(0)) > 0)
    & (F.coalesce(F.col("VL_TRANS_SAI"), F.lit(0)) > 0)
)

perfil_renda_valido = ~texto_normalizado("NM_PRFL").isin("A CLASSIFICAR", "SEM PERFIL")
macroperfil_valido = texto_normalizado("NM_MAC_PRFL_CLI") != "A CLASSIFICAR"
microperfil_valido = texto_normalizado("NM_MIC_PRFL_CLI") != "A CLASSIFICAR"
perfil_financeiro_valido = texto_normalizado("NM_PRFL_FIN") != "A CLASSIFICAR"

perfil_completo = (
    perfil_renda_valido
    & macroperfil_valido
    & microperfil_valido
    & perfil_financeiro_valido
)

sem_agro = texto_normalizado("FL_TEM_MOV_AGRO", "N") == "N"
participa_radar = movimentacao_minima & perfil_completo & sem_agro

df_base_estudo = (
    df
    .withColumn("FL_MOV_MINIMA", F.when(movimentacao_minima, "S").otherwise("N"))
    .withColumn("FL_PERFIL_RENDA_VALIDO", F.when(perfil_renda_valido, "S").otherwise("N"))
    .withColumn("FL_MACROPERFIL_VALIDO", F.when(macroperfil_valido, "S").otherwise("N"))
    .withColumn("FL_MICROPERFIL_VALIDO", F.when(microperfil_valido, "S").otherwise("N"))
    .withColumn("FL_PERFIL_FIN_VALIDO", F.when(perfil_financeiro_valido, "S").otherwise("N"))
    .withColumn("FL_PERFIL_COMPLETO", F.when(perfil_completo, "S").otherwise("N"))
    .withColumn("FL_SEM_AGRO", F.when(sem_agro, "S").otherwise("N"))
    .withColumn("FL_PARTICIPA_RADAR_ESTUDO", F.when(participa_radar, "S").otherwise("N"))
    .withColumn(
        "TX_PARTICIPACAO_RADAR",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S", "Participa")
         .otherwise("Nao participa")
    )
    .withColumn("TX_PRFL_ESTUDO", F.coalesce(F.col("NM_PRFL").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_MACROPERFIL_ESTUDO", F.coalesce(F.col("NM_MAC_PRFL_CLI").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_MICROPERFIL_ESTUDO", F.coalesce(F.col("NM_MIC_PRFL_CLI").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_PERFIL_FIN_ESTUDO", F.coalesce(F.col("NM_PRFL_FIN").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_RES_ORC_ESTUDO", F.coalesce(F.col("TX_RES_ORC").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_STS_FINAL_ESTUDO", F.coalesce(F.col("TX_STS_FINAL").cast("string"), F.lit("Nao informado")))
    .withColumn(
        "ORDEM_RES_ORC",
        F.when(F.col("CD_RES_ORC") == 1, 10)
         .when(F.col("CD_RES_ORC") == 0, 20)
         .when(F.col("CD_RES_ORC") == 2, 30)
         .otherwise(99)
    )
    .withColumn(
        "ORDEM_STS_FINAL",
        F.when((F.col("CD_RES_ORC") == 1) & (texto_normalizado("TX_STS_RES", "") == "FORTE"), 10)
         .when((F.col("CD_RES_ORC") == 1) & (texto_normalizado("TX_STS_RES", "") == "FRACO"), 20)
         .when((F.col("CD_RES_ORC") == 0) & (texto_normalizado("TX_STS_RES", "") == "FRACO"), 30)
         .when((F.col("CD_RES_ORC") == 0) & (texto_normalizado("TX_STS_RES", "") == "FORTE"), 40)
         .when((F.col("CD_RES_ORC") == 2) & (texto_normalizado("TX_STS_RES", "") == "FRACO"), 50)
         .when((F.col("CD_RES_ORC") == 2) & (texto_normalizado("TX_STS_RES", "") == "FORTE"), 60)
         .otherwise(99)
    )
)
df_base_estudo = (
    df_base_estudo
    .withColumn(
        "NR_MAIOR_PONTUACAO",
        F.greatest(*[F.coalesce(F.col(coluna).cast("int"), F.lit(0)) for _, coluna, _ in temas_pontuacao])
    )
    .withColumn(
        "ARR_TEMAS_VENCEDORES_RAW",
        F.array(*[
            F.when(
                (F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S")
                & (F.col("NR_MAIOR_PONTUACAO") > 0)
                & (F.coalesce(F.col(coluna).cast("int"), F.lit(0)) == F.col("NR_MAIOR_PONTUACAO")),
                F.lit(tema)
            )
            for tema, coluna, _ in temas_pontuacao
        ])
    )
    .withColumn("ARR_TEMAS_VENCEDORES", F.expr("filter(ARR_TEMAS_VENCEDORES_RAW, x -> x is not null)"))
    .withColumn("QT_TEMAS_VENCEDORES", F.size(F.col("ARR_TEMAS_VENCEDORES")))
    .withColumn(
        "TX_TEMAS_VENCEDORES",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N", "Nao participa")
         .when(F.col("QT_TEMAS_VENCEDORES") == 0, "Sem prioridade")
         .otherwise(F.array_join(F.col("ARR_TEMAS_VENCEDORES"), " + "))
    )
    .withColumn(
        "TX_TEMA_VENCEDOR_UNICO",
        F.when(
            F.col("QT_TEMAS_VENCEDORES") == 1,
            F.element_at(F.col("ARR_TEMAS_VENCEDORES"), 1)
        )
    )
    .withColumn(
        "TX_RESULTADO_FINAL_RADAR",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N", "Nao participa")
         .when(F.col("NR_MAIOR_PONTUACAO") == 0, "Sem prioridade")
         .when(F.col("QT_TEMAS_VENCEDORES") == 1, "Tema prioritario unico")
         .otherwise("Empate no topo")
    )
)

qt_base = df_base_estudo.select(F.countDistinct("CD_CLI").alias("QT_BASE")).first()["QT_BASE"]
qt_participantes = (
    df_base_estudo
    .where(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S")
    .select(F.countDistinct("CD_CLI").alias("QT"))
    .first()["QT"]
)
qt_nao_participantes = qt_base - qt_participantes

linha_periodo = df_base_estudo.select(
    F.min("DT_REF_INI").alias("DT_REF_INI"),
    F.max("DT_REF_FIM").alias("DT_REF_FIM")
).first()

df_participantes = df_base_estudo.where(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S")
df_vencedor_unico = df_participantes.where(F.col("TX_RESULTADO_FINAL_RADAR") == "Tema prioritario unico")
df_empates = df_participantes.where(F.col("TX_RESULTADO_FINAL_RADAR") == "Empate no topo")

df_base_estudo.createOrReplaceTempView("vw_base_estudo_radar")

print(f"Fonte dos dados: {tabela_origem}")
print(f"Clientes analisados no periodo: {qt_base}")
print(f"Clientes aptos a participar do Radar: {qt_participantes}")
print(f"Clientes fora do Radar: {qt_nao_participantes}")
print(f"Periodo analisado: {linha_periodo['DT_REF_INI']} a {linha_periodo['DT_REF_FIM']}")


## 1. Participacao no Radar

Cada card mostra uma regra obrigatoria do Radar e a quantidade de clientes que nao participa por aquela regra. Um mesmo cliente pode aparecer em mais de uma regra.

In [ ]:
%%spark

print("1.1 Regras que definem quem participa do Radar")
print("Leitura: cada card mostra a quantidade de clientes que nao participa do Radar por nao cumprir aquela regra.")
print("Observacao: um mesmo cliente pode aparecer em mais de uma regra.")

linhas_participacao = [
    F.struct(
        F.lit(10).alias("ORDEM"),
        F.lit("Movimentacao do cliente").alias("Bloco"),
        F.lit("Teve pelo menos uma transacao no periodo analisado").alias("Regra"),
        F.when(F.coalesce(F.col("QT_TRANS_TOTAL"), F.lit(0)) > 0, "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.coalesce(F.col("QT_TRANS_TOTAL"), F.lit(0)) > 0,
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque nao teve transacao no periodo analisado").alias("Leitura")
    ),
    F.struct(
        F.lit(20).alias("ORDEM"),
        F.lit("Movimentacao do cliente").alias("Bloco"),
        F.lit("Teve pelo menos uma transacao de entrada").alias("Regra"),
        F.when(F.coalesce(F.col("QT_TRANS_ENT"), F.lit(0)) > 0, "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.coalesce(F.col("QT_TRANS_ENT"), F.lit(0)) > 0,
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque nao teve transacao de entrada").alias("Leitura")
    ),
    F.struct(
        F.lit(30).alias("ORDEM"),
        F.lit("Movimentacao do cliente").alias("Bloco"),
        F.lit("Teve pelo menos uma transacao de saida").alias("Regra"),
        F.when(F.coalesce(F.col("QT_TRANS_SAI"), F.lit(0)) > 0, "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.coalesce(F.col("QT_TRANS_SAI"), F.lit(0)) > 0,
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque nao teve transacao de saida").alias("Leitura")
    ),
    F.struct(
        F.lit(40).alias("ORDEM"),
        F.lit("Movimentacao do cliente").alias("Bloco"),
        F.lit("Teve valor financeiro de entrada maior que zero").alias("Regra"),
        F.when(F.coalesce(F.col("VL_TRANS_ENT"), F.lit(0)) > 0, "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.coalesce(F.col("VL_TRANS_ENT"), F.lit(0)) > 0,
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque o valor de entrada foi zero ou nao informado").alias("Leitura")
    ),
    F.struct(
        F.lit(50).alias("ORDEM"),
        F.lit("Movimentacao do cliente").alias("Bloco"),
        F.lit("Teve valor financeiro de saida maior que zero").alias("Regra"),
        F.when(F.coalesce(F.col("VL_TRANS_SAI"), F.lit(0)) > 0, "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.coalesce(F.col("VL_TRANS_SAI"), F.lit(0)) > 0,
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque o valor de saida foi zero ou nao informado").alias("Leitura")
    ),
    F.struct(
        F.lit(60).alias("ORDEM"),
        F.lit("Perfil do cliente").alias("Bloco"),
        F.lit("Tem perfil de renda identificado, diferente de A CLASSIFICAR e SEM PERFIL").alias("Regra"),
        F.when(F.col("FL_PERFIL_RENDA_VALIDO") == "S", "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.col("FL_PERFIL_RENDA_VALIDO") == "S",
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque o perfil de renda esta A CLASSIFICAR, SEM PERFIL ou nao informado").alias("Leitura")
    ),
    F.struct(
        F.lit(70).alias("ORDEM"),
        F.lit("Perfil do cliente").alias("Bloco"),
        F.lit("Tem macroperfil identificado").alias("Regra"),
        F.when(F.col("FL_MACROPERFIL_VALIDO") == "S", "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.col("FL_MACROPERFIL_VALIDO") == "S",
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque o macroperfil nao esta identificado").alias("Leitura")
    ),
    F.struct(
        F.lit(80).alias("ORDEM"),
        F.lit("Perfil do cliente").alias("Bloco"),
        F.lit("Tem microperfil identificado").alias("Regra"),
        F.when(F.col("FL_MICROPERFIL_VALIDO") == "S", "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.col("FL_MICROPERFIL_VALIDO") == "S",
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque o microperfil nao esta identificado").alias("Leitura")
    ),
    F.struct(
        F.lit(90).alias("ORDEM"),
        F.lit("Perfil do cliente").alias("Bloco"),
        F.lit("Tem perfil financeiro consolidado").alias("Regra"),
        F.when(F.col("FL_PERFIL_FIN_VALIDO") == "S", "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.col("FL_PERFIL_FIN_VALIDO") == "S",
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque o perfil financeiro consolidado nao esta identificado").alias("Leitura")
    ),
    F.struct(
        F.lit(100).alias("ORDEM"),
        F.lit("Movimentacao Agro").alias("Bloco"),
        F.lit("Nao teve movimentacao marcada como Agro").alias("Regra"),
        F.when(F.col("FL_SEM_AGRO") == "S", "Cumpre a regra").otherwise("Nao cumpre a regra").alias("Situacao"),
        F.when(
            F.col("FL_SEM_AGRO") == "S",
            "Esta regra nao impede a participacao do cliente no Radar"
        ).otherwise("Nao participa do Radar porque teve movimentacao marcada como Agro").alias("Leitura")
    ),
    F.struct(
        F.lit(110).alias("ORDEM"),
        F.lit("Resultado de participacao").alias("Bloco"),
        F.lit("Cliente cumpre movimento, perfil e regra Agro para entrar no Radar").alias("Regra"),
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S", "Participa do Radar").otherwise("Nao participa do Radar").alias("Situacao"),
        F.when(
            F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S",
            "Cliente cumpre todas as regras obrigatorias e entra no Radar"
        ).otherwise("Cliente nao participa do Radar porque nao cumpriu pelo menos uma regra obrigatoria").alias("Leitura")
    ),
]

df_participacao_radar = (
    df_base_estudo
    .select("CD_CLI", F.explode(F.array(*linhas_participacao)).alias("R"))
    .select(
        "CD_CLI",
        F.col("R.ORDEM").alias("ORDEM"),
        F.col("R.Bloco").alias("Bloco"),
        F.col("R.Regra").alias("Regra"),
        F.col("R.Situacao").alias("Situacao"),
        F.col("R.Leitura").alias("Leitura")
    )
)

df_regras_participacao = (
    df_participacao_radar
    .select("ORDEM", "Bloco", "Regra")
    .distinct()
)

df_clientes_fora_por_regra = (
    df_participacao_radar
    .where(F.col("Situacao").isin("Nao cumpre a regra", "Nao participa do Radar"))
    .groupBy("ORDEM", "Bloco", "Regra")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
)

df_cards_regras_participacao = (
    df_regras_participacao
    .join(df_clientes_fora_por_regra, ["ORDEM", "Bloco", "Regra"], "left")
    .fillna(0, subset=["Clientes"])
    .orderBy("ORDEM")
)

cards_regras_participacao = (
    df_cards_regras_participacao
    .select("ORDEM", "Bloco", "Regra")
    .orderBy("ORDEM")
    .collect()
)

for card in cards_regras_participacao:
    print("")
    print(f"CARD - {card['Bloco']}: {card['Regra']}")
    mostrar(
        df_cards_regras_participacao
        .where(F.col("ORDEM") == card["ORDEM"])
        .select("Clientes")
    )


## 2. Resultado final e temas prioritarios

Estudo global da escolha final do Radar: tema prioritario, empates em primeiro lugar e clientes sem prioridade definida.

In [ ]:
%%spark

print("2.1 Como o Radar escolhe o tema prioritario")
df_regra_temas = spark.createDataFrame(
    [(tema, "Compara a pontuacao final deste tema com a dos demais temas") for tema, coluna, _ in temas_pontuacao],
    ["Tema avaliado", "Criterio de escolha"]
)
mostrar(df_regra_temas)
print("Leitura: tema prioritario unico = um tema ficou sozinho com a maior pontuacao; empate = dois ou mais temas ficaram em primeiro lugar; sem prioridade = nenhum tema teve pontuacao relevante.")

qt_vencedor_unico = df_vencedor_unico.select(F.countDistinct("CD_CLI").alias("QT")).first()["QT"]
qt_empates = df_empates.select(F.countDistinct("CD_CLI").alias("QT")).first()["QT"]

print("2.2 Resumo da escolha final do Radar")
df_resumo_decisao = (
    contar_clientes(
        df_participantes,
        ["TX_RESULTADO_FINAL_RADAR"],
        qt_participantes,
        "% Participantes"
    )
    .withColumn(
        "ORDEM",
        F.when(F.col("TX_RESULTADO_FINAL_RADAR") == "Tema prioritario unico", 10)
         .when(F.col("TX_RESULTADO_FINAL_RADAR") == "Empate no topo", 20)
         .when(F.col("TX_RESULTADO_FINAL_RADAR") == "Sem prioridade", 30)
         .otherwise(99)
    )
    .orderBy("ORDEM")
    .select(F.col("TX_RESULTADO_FINAL_RADAR").alias("Leitura final do Radar"), "Clientes", "% Participantes")
)
mostrar(df_resumo_decisao)

print("2.3 Quantidade de temas empatados em primeiro lugar")
df_qtd_temas_topo = (
    contar_clientes(
        df_participantes,
        ["QT_TEMAS_VENCEDORES"],
        qt_participantes,
        "% Participantes",
        ["QT_TEMAS_VENCEDORES"]
    )
    .withColumn(
        "Qtd temas no topo",
        F.concat(
            F.col("QT_TEMAS_VENCEDORES").cast("string"),
            F.when(F.col("QT_TEMAS_VENCEDORES") == 1, F.lit(" tema")).otherwise(F.lit(" temas"))
        )
    )
    .select("Qtd temas no topo", "Clientes", "% Participantes")
)
mostrar(df_qtd_temas_topo)

print("2.4 Temas prioritarios quando nao houve empate")
df_temas_vencedores_unicos = (
    contar_clientes(
        df_vencedor_unico,
        ["TX_TEMA_VENCEDOR_UNICO"],
        qt_participantes,
        "% Participantes",
        [F.desc("Clientes")]
    )
    .withColumn("% Vencedor unico", pct_sobre("Clientes", qt_vencedor_unico))
    .select(F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema prioritario"), "Clientes", "% Participantes", "% dos temas prioritarios unicos")
)
mostrar(df_temas_vencedores_unicos)

print("2.5 Temas que ficaram em primeiro lugar, incluindo empates")
df_temas_no_topo = (
    df_participantes
    .where(F.col("QT_TEMAS_VENCEDORES") > 0)
    .select("CD_CLI", F.explode("ARR_TEMAS_VENCEDORES").alias("Tema em primeiro lugar"))
)

df_temas_no_topo_exibicao = (
    contar_clientes(df_temas_no_topo, ["Tema em primeiro lugar"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select("Tema em primeiro lugar", "Clientes", "% Participantes")
)
mostrar(df_temas_no_topo_exibicao)
print("Observacao: quando ha empate, o cliente aparece em todos os temas que dividiram o primeiro lugar.")

print("2.6 Principais empates entre temas")
df_top3_empates = (
    contar_clientes(
        df_empates,
        ["TX_TEMAS_VENCEDORES"],
        qt_participantes,
        "% Participantes",
        [F.desc("Clientes")]
    )
    .withColumn("% Empates", pct_sobre("Clientes", qt_empates))
    .select(F.col("TX_TEMAS_VENCEDORES").alias("Temas empatados em primeiro lugar"), "Clientes", "% Participantes", "% dos empates")
    .limit(3)
)
mostrar(df_top3_empates)


## 3. Resultado financeiro x temas prioritarios

Estudo do resultado financeiro do mes, em visao geral e por intensidade, conectado aos temas prioritarios sem empate.

In [ ]:
%%spark

print("3.1 Como o Radar classifica o resultado financeiro do mes")
df_regra_orcamento = spark.createDataFrame(
    [
        ("Indicador usado", "Compara o total de saidas com o total de entradas do cliente"),
        ("Saidas abaixo de 75% das entradas", "Superavitario Forte"),
        ("Saidas de 75% ate menos de 95% das entradas", "Superavitario Fraco"),
        ("Saidas de 95% ate menos de 100% das entradas", "Neutro Fraco"),
        ("Saidas de 100% ate 105% das entradas", "Neutro Forte"),
        ("Saidas acima de 105% ate 125% das entradas", "Deficitario Fraco"),
        ("Saidas acima de 125% das entradas", "Deficitario Forte"),
    ],
    ["Faixa de leitura", "Classificacao do mes"]
)
mostrar(df_regra_orcamento)

print("3.2 Resultado financeiro do mes - visao geral")
df_orcamento_macro = (
    df_participantes
    .groupBy("TX_RES_ORC_ESTUDO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_RES_ORC").alias("ORDEM")
    )
    .withColumn("% Participantes", pct_sobre("Clientes", qt_participantes))
    .orderBy("ORDEM")
    .select(F.col("TX_RES_ORC_ESTUDO").alias("Resultado financeiro"), "Clientes", "% Participantes")
)
mostrar(df_orcamento_macro)

print("3.3 Resultado financeiro do mes - intensidade")
df_orcamento_micro = (
    df_participantes
    .groupBy("TX_STS_FINAL_ESTUDO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_STS_FINAL").alias("ORDEM")
    )
    .withColumn("% Participantes", pct_sobre("Clientes", qt_participantes))
    .orderBy("ORDEM")
    .select(F.col("TX_STS_FINAL_ESTUDO").alias("Leitura do resultado financeiro"), "Clientes", "% Participantes")
)
mostrar(df_orcamento_micro)

print("3.4 Resultado financeiro geral x tema prioritario")
df_orc_macro_tema = (
    df_vencedor_unico
    .groupBy("TX_RES_ORC_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_RES_ORC").alias("ORDEM")
    )
    .withColumn(
        "% do resultado",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_RES_ORC_ESTUDO")) * 100, 2)
    )
    .orderBy("ORDEM", F.desc("Clientes"))
    .select(
        F.col("TX_RES_ORC_ESTUDO").alias("Resultado financeiro"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema prioritario"),
        "Clientes",
        "% do resultado"
    )
)
mostrar(df_orc_macro_tema)

print("3.5 Intensidade do resultado financeiro x tema prioritario")
df_orc_micro_tema = (
    df_vencedor_unico
    .groupBy("TX_STS_FINAL_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_STS_FINAL").alias("ORDEM")
    )
    .withColumn(
        "% do status",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_STS_FINAL_ESTUDO")) * 100, 2)
    )
    .orderBy("ORDEM", F.desc("Clientes"))
    .select(
        F.col("TX_STS_FINAL_ESTUDO").alias("Leitura do resultado financeiro"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema prioritario"),
        "Clientes",
        "% do status"
    )
)
mostrar(df_orc_micro_tema)
print("Observacao: as tabelas de conexao com tema prioritario consideram apenas clientes sem empate entre temas.")


## 4. Perfil x temas prioritarios

Estudo de perfil de renda, macroperfil, microperfil e perfil financeiro, conectado aos temas prioritarios sem empate.

In [ ]:
%%spark

print("4.1 Quando o perfil do cliente e considerado completo")
df_regra_perfil = spark.createDataFrame(
    [
        ("Perfil de renda", "Precisa estar identificado e nao pode ser A CLASSIFICAR nem SEM PERFIL"),
        ("Macroperfil", "Precisa estar identificado"),
        ("Microperfil", "Precisa estar identificado"),
        ("Perfil financeiro consolidado", "Precisa estar identificado"),
    ],
    ["Informacao do cliente", "Criterio para participar do Radar"]
)
mostrar(df_regra_perfil)

print("4.2 Perfil de renda dos clientes aptos ao Radar")
df_perfil_renda = (
    contar_clientes(df_participantes, ["TX_PRFL_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_PRFL_ESTUDO").alias("Perfil de renda"), "Clientes", "% Participantes")
)
mostrar(df_perfil_renda)

print("4.3 Macroperfil dos clientes aptos ao Radar")
df_macroperfil = (
    contar_clientes(df_participantes, ["TX_MACROPERFIL_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_MACROPERFIL_ESTUDO").alias("Macroperfil"), "Clientes", "% Participantes")
)
mostrar(df_macroperfil)

print("4.4 Microperfil dos clientes aptos ao Radar")
df_microperfil = (
    contar_clientes(df_participantes, ["TX_MICROPERFIL_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_MICROPERFIL_ESTUDO").alias("Microperfil"), "Clientes", "% Participantes")
)
mostrar(df_microperfil)

print("4.5 Perfil financeiro dos clientes aptos ao Radar")
df_perfil_financeiro = (
    contar_clientes(df_participantes, ["TX_PERFIL_FIN_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_PERFIL_FIN_ESTUDO").alias("Perfil financeiro"), "Clientes", "% Participantes")
)
mostrar(df_perfil_financeiro)

print("4.6 Macroperfil x tema prioritario")
df_macro_tema = (
    df_vencedor_unico
    .groupBy("TX_MACROPERFIL_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
    .withColumn(
        "% do macroperfil",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_MACROPERFIL_ESTUDO")) * 100, 2)
    )
    .orderBy("TX_MACROPERFIL_ESTUDO", F.desc("Clientes"))
    .select(
        F.col("TX_MACROPERFIL_ESTUDO").alias("Macroperfil"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema prioritario"),
        "Clientes",
        "% do macroperfil"
    )
)
mostrar(df_macro_tema)

print("4.7 Microperfil x tema prioritario")
df_micro_tema = (
    df_vencedor_unico
    .groupBy("TX_MICROPERFIL_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
    .withColumn(
        "% do microperfil",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_MICROPERFIL_ESTUDO")) * 100, 2)
    )
    .orderBy("TX_MICROPERFIL_ESTUDO", F.desc("Clientes"))
    .select(
        F.col("TX_MICROPERFIL_ESTUDO").alias("Microperfil"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema prioritario"),
        "Clientes",
        "% do microperfil"
    )
)
mostrar(df_micro_tema)

print("4.8 Perfil financeiro x tema prioritario")
df_perfil_fin_tema = (
    df_vencedor_unico
    .groupBy("TX_PERFIL_FIN_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
    .withColumn(
        "% do perfil",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_PERFIL_FIN_ESTUDO")) * 100, 2)
    )
    .orderBy("TX_PERFIL_FIN_ESTUDO", F.desc("Clientes"))
    .select(
        F.col("TX_PERFIL_FIN_ESTUDO").alias("Perfil financeiro"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema prioritario"),
        "Clientes",
        "% do perfil"
    )
)
mostrar(df_perfil_fin_tema)
print("Observacao: as tabelas de conexao com tema prioritario consideram apenas clientes sem empate entre temas.")
